# SC203 DoSSR — Baseline, VAE SDPA Memory Profile & Seed Robustness Study (`01_dossr_baseline.ipynb`)

**Cell Type:** `MARKDOWN`  
**Team:** LEMMEPASS (Team 1, SC203 Scientific Methods — Class 25A02)  
**Target Branch:** `feat/baseline-runner`  
**Verification Status:** **`NOT TESTED`** (This notebook and the VAE `AttnBlock` SDPA patch have not been executed on Google Colab yet.)

---

## 1. Experiment Overview
- **Objective:** Verify the runtime PyTorch Scaled Dot-Product Attention (SDPA) patch for the VAE `AttnBlock` (`src/attention_compat.py`), measure 1-image FP16 peak VRAM against the pre-patch `36.04 GiB` baseline, verify seed sensitivity against a same-seed repeat control (`seed 42` vs `seed 42-repeat` vs `seed 43`), and run the 5-image / 5-seed (`42..46`) RealSR study.
- **Scope & Baseline vs. Modifications:**
  1. **Original DoSSR Baseline (`external/DOSSR`):** Untouched upstream code and `dossr_default.ckpt` checkpoint (`steps=5`, `solver_order=3`, `sr_scale=4`, `color_fix_type=wavelet`, `preprocessor=RealESRNet`).
  2. **Runtime Compatibility Layer (`src/dossr_inference_compat.py` + `src/attention_compat.py`):**
     - Skips the missing author-local OpenCLIP initialization path (all parameters come from the strict `dossr_default.ckpt` load).
     - Loads checkpoint with `torch.load(..., mmap=True)`.
     - Enables `torch.autocast("cuda", dtype=torch.float16, cache_enabled=False)` when `--fp16` (`precision: fp16`) is passed.
     - Replaces vanilla `CrossAttention.forward` (`einsum`) with PyTorch `scaled_dot_product_attention` when `xformers` is absent.
     - Replaces vanilla VAE `AttnBlock.forward` (`bmm`) with single-head PyTorch `scaled_dot_product_attention` (`scale = channels ** -0.5`), retaining `norm`, `q/k/v` projections, `proj_out`, residual connection, and fallback to `bmm`. (**`Status: NOT TESTED` on Colab**).
- **Hardware Requirements:**
  - Google Colab GPU runtime with CUDA.
  - **VRAM Note:** Before the VAE `AttnBlock` patch, 1-image FP16 peak allocated VRAM was `36.04 GiB` (`52.70 GiB` reserved). `CELL 03` warns if total GPU VRAM is below `40.0 GiB`.
- **Execution Order:**
  - Run `CELL 01` through `CELL 11` sequentially in Google Colab.
  - `CELL 12` (5-image batch) and `CELL 13` (5-seed study) are gated by `RUN_SEED_STUDY = False` (set in `CELL 01`) and by on-disk JSON status files written by `CELL 10` and `CELL 11`. Set `RUN_SEED_STUDY = True` only after reviewing `CELL 09`–`CELL 11` logs.


---
## 2. Environment Setup & Configuration (`CELL 01`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Mounts Google Drive, sets configurable directory paths (`ASSETS_DIR`, `REPO_DIR`, `OUTPUT_BASE`), defines workflow toggles (`CACHE_CKPT_TO_LOCAL_SSD = False`, `RUN_SEED_STUDY = False`, `FORCE_SETUP = False`, `SEED_SPREAD_MIN_RATIO = 3.0`), and defines the two-pass `clean_corrupt_pngs()` helper.
- **Why it is necessary:** Establishes portable paths across all team members and ensures output folders are saved on Google Drive (`OUTPUT_BASE`) so finished images persist across disconnects.
- **Side effects:** Mounts `/content/drive` and creates log/output directories on Drive if absent. Does not consume GPU resources.
- **Expected output:** Prints mounted paths, toggle states, and `CELL 01 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 01 [IN-CELL RUN] — Mount Google Drive, configure paths/flags, and define helpers (Status: NOT TESTED)
from datetime import datetime, timezone
from pathlib import Path
from PIL import Image

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as exc:
    raise RuntimeError("This notebook must be executed inside Google Colab.") from exc

# Configurable paths (adjust ASSETS_DIR only if your shared Drive folder is mounted elsewhere)
BRANCH = "feat/baseline-runner"
REPO_URL = "https://github.com/lemon4life/SC203.git"
REPO_DIR = Path("/content/SC203")
ASSETS_DIR = Path("/content/drive/MyDrive/lemmepass")
LOG_DIR = ASSETS_DIR / "logs"
OUTPUT_BASE = ASSETS_DIR / "outputs"
STATUS_DIR = Path("/content/sc203_status")

# Workflow toggles
CACHE_CKPT_TO_LOCAL_SSD = False  # Default False: OS page cache already makes 2nd load fast (~1.0s)
RUN_SEED_STUDY = False           # Keep False until CELL 09-11 are reviewed and verified
FORCE_SETUP = False              # Set True to force re-running colab_setup.py
SEED_SPREAD_MIN_RATIO = 3.0      # Required ratio: mean_abs_diff(42 vs 43) >= 3.0 * mean_abs_diff(42 vs 42-repeat)

LOG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
STATUS_DIR.mkdir(parents=True, exist_ok=True)

CELL10_STATUS_JSON = STATUS_DIR / "cell10_inspection_status.json"
CELL11_STATUS_JSON = STATUS_DIR / "cell11_seed_sanity_status.json"


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def clean_corrupt_pngs(current_output_folder: Path) -> list[str]:
    """Inspect only current_output_folder; open each PNG twice (verify + load) and delete corrupt files."""
    removed: list[str] = []
    if not current_output_folder.exists():
        return removed
    for png_path in sorted(current_output_folder.rglob("*.png")):
        if not png_path.is_file():
            continue
        try:
            with Image.open(png_path) as img_verify:
                img_verify.verify()
            with Image.open(png_path) as img_load:
                img_load.load()
        except Exception as err:
            print(f"[CORRUPT PNG REMOVED] {png_path} ({type(err).__name__}: {err})")
            png_path.unlink(missing_ok=True)
            removed.append(str(png_path))
    print(f"Checked PNGs in {current_output_folder}: removed {len(removed)} corrupt file(s).")
    return removed


print(f"ASSETS_DIR              : {ASSETS_DIR}")
print(f"LOG_DIR                 : {LOG_DIR}")
print(f"OUTPUT_BASE             : {OUTPUT_BASE}")
print(f"CACHE_CKPT_TO_LOCAL_SSD : {CACHE_CKPT_TO_LOCAL_SSD}")
print(f"RUN_SEED_STUDY          : {RUN_SEED_STUDY}")
print(f"SEED_SPREAD_MIN_RATIO   : {SEED_SPREAD_MIN_RATIO}")
print("CELL 01 READY")


---
## 3. Repository Checkout & Commit Provenance (`CELL 02`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Clones `https://github.com/lemon4life/SC203.git` into `/content/SC203` (or fetches and fast-forwards if already cloned), checks out `BRANCH`, and dynamically queries `git rev-parse HEAD`.
- **Why it is necessary:** Ensures the Colab VM runs the latest committed code and records the exact commit hash for reproducibility and status file checks.
- **Dependencies:** Requires `CELL 01`. Modifies `/content/SC203`. No GPU usage.
- **Expected output:** Prints the full and short commit hashes dynamically and `CELL 02 READY`.
- **Troubleshooting:** If `git pull --ff-only` fails due to local edits inside `/content/SC203`, reset `/content/SC203` or restart the runtime.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 02 [IN-CELL RUN] — Clone or fast-forward repository and print commit hash (Status: NOT TESTED)
import subprocess

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

GIT_COMMIT_FULL = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
GIT_COMMIT_SHORT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print(f"Repository : {REPO_DIR}")
print(f"Branch     : {BRANCH}")
print(f"Commit     : {GIT_COMMIT_SHORT} ({GIT_COMMIT_FULL})")
print("CELL 02 READY")


---
## 4. Hardware, GPU, CUDA & VRAM Verification (`CELL 03`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Runs `nvidia-smi` and `free -h`, checks PyTorch CUDA and `scaled_dot_product_attention` availability, reports GPU name and total/free VRAM in GiB, and warns if total GPU VRAM is below `40.0 GiB`.
- **Why it is necessary:** Before the VAE `AttnBlock` patch, 1-image FP16 inference peaked at `36.04 GiB` allocated (`52.70 GiB` reserved). Until the new VAE `AttnBlock` SDPA patch is profiled on Colab, runtimes with `< 40.0 GiB` VRAM may run out of memory.
- **Dependencies:** Requires a GPU runtime in Colab (`Runtime -> Change runtime type -> GPU`).
- **Expected output:** GPU model name, total/free GiB, SDPA availability `True`, and `CELL 03 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 03 [IN-CELL RUN] — Verify GPU, CUDA, SDPA, and VRAM capacity (Status: NOT TESTED)
import platform
import subprocess
import sys
import torch
import torch.nn.functional as F

print("=== nvidia-smi ===")
subprocess.run(["nvidia-smi"], check=True)
print("\n=== System RAM (free -h) ===")
subprocess.run(["free", "-h"], check=True)

print("\n=== Python & PyTorch ===")
print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("SDPA available:", hasattr(F, "scaled_dot_product_attention"))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Select a GPU runtime in Colab before continuing.")
if not hasattr(F, "scaled_dot_product_attention"):
    raise RuntimeError("PyTorch scaled_dot_product_attention is required.")

props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
total_gib = total_bytes / (2**30)
free_gib = free_bytes / (2**30)
print(f"GPU 0   : {props.name} | total={total_gib:.2f} GiB | free={free_gib:.2f} GiB")

VRAM_WARNING_GIB = 40.0
if total_gib < VRAM_WARNING_GIB:
    print(
        f"WARNING: GPU total VRAM ({total_gib:.2f} GiB) is below {VRAM_WARNING_GIB:.1f} GiB. "
        "Pre-VAE-patch FP16 peak allocated VRAM was 36.04 GiB (52.70 GiB reserved); "
        "post-VAE-patch VRAM reduction is NOT TESTED yet."
    )
print("CELL 03 READY")


---
## 5. Dataset Structure & Input Resolution Audit (`CELL 04`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Lists all subdirectories under `ASSETS_DIR / "data" / "StableSR_testsets"` (to check whether ground-truth HR folders exist), counts image files in `RealSRVal_crop128`, and inspects the first image (`Canon_001_LR4.png`) to print its exact pixel dimensions (`width x height`) and mode.
- **Why it is necessary:** Resolves the open questions in `context.md` regarding HR reference availability, exact image count, and why `sr_scale: 4` produces `2048x2048` outputs.
- **Dependencies:** Requires `CELL 01` (mounted Google Drive with Phase 1 assets). Read-only.
- **Expected output:** Directory listing under `StableSR_testsets`, image count in `RealSRVal_crop128`, first image dimensions, and `CELL 04 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 04 [IN-CELL RUN] — Inspect StableSR_testsets folders and input image size (Status: NOT TESTED)
from PIL import Image

TESTSETS_ROOT = ASSETS_DIR / "data" / "StableSR_testsets"
REALSR_DIR = TESTSETS_ROOT / "StableSR_testsets" / "RealSRVal_crop128"
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

if not TESTSETS_ROOT.is_dir():
    raise FileNotFoundError(f"Missing dataset root on Drive: {TESTSETS_ROOT}")

print("=== Directories under StableSR_testsets (max depth 3) ===")
for folder in sorted(TESTSETS_ROOT.rglob("*")):
    if folder.is_dir() and len(folder.relative_to(TESTSETS_ROOT).parts) <= 3:
        imgs = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        print(f"  {folder.relative_to(TESTSETS_ROOT)} (direct images: {len(imgs)})")

if not REALSR_DIR.is_dir():
    raise FileNotFoundError(f"Missing RealSR directory: {REALSR_DIR}")

realsr_images = sorted(
    p for p in REALSR_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)
print(f"\nRealSRVal_crop128 total images: {len(realsr_images)}")
if not realsr_images:
    raise RuntimeError(f"No images found in {REALSR_DIR}")

first_img_path = realsr_images[0]
with Image.open(first_img_path) as im:
    print(f"First image : {first_img_path.name} | size={im.size} (WxH) | mode={im.mode}")
print("CELL 04 READY")


---
## 6. Guarded Dependency Installation (`CELL 05`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Uses `importlib.util.find_spec` to check whether DoSSR's required packages (`pytorch_lightning`, `open_clip`, `omegaconf`, `lpips`, `einops`, `cv2`, `scipy`, `transformers`, `facexlib`) are already installed in the current runtime, and verifies `external/DOSSR/scripts/inference.py` exists. If any spec is missing, if `external/DOSSR` is absent, or if `FORCE_SETUP = True`, it installs `jedi==0.20.1` and runs `scripts/colab_setup.py`.
- **Why it is necessary:** Installs DoSSR's dependencies and clones `external/DOSSR` without modifying Colab's preinstalled `torch` and `torchvision` packages, and avoids redundant reinstallations if the runtime is still warm.
- **Dependencies:** Requires `CELL 01` and `CELL 02`. Modifies Python packages and `/content/SC203/external/DOSSR` on a cold runtime. No GPU usage.
- **Expected output:** Lists checked module specs, runs `colab_setup.py` if needed, and prints `CELL 05 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 05 [IN-CELL RUN] — Check module specs via find_spec and run guarded setup if needed (Status: NOT TESTED)
import importlib.util
import subprocess
import sys

REQUIRED_SPECS = (
    "pytorch_lightning",
    "open_clip",
    "omegaconf",
    "lpips",
    "einops",
    "cv2",
    "scipy",
    "transformers",
    "facexlib",
)
missing_specs = [name for name in REQUIRED_SPECS if importlib.util.find_spec(name) is None]
dossr_inference_py = REPO_DIR / "external" / "DOSSR" / "scripts" / "inference.py"

print("Missing module specs:", missing_specs)
print("DoSSR inference.py present:", dossr_inference_py.is_file())

if FORCE_SETUP or missing_specs or not dossr_inference_py.is_file():
    print("Running Jedi repair and scripts/colab_setup.py ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "jedi==0.20.1"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts" / "colab_setup.py"),
            "--requirements",
            str(REPO_DIR / "external" / "DOSSR" / "requirements.txt"),
            "--log-dir",
            str(LOG_DIR),
        ],
        check=True,
    )
else:
    print("All required module specs and external/DOSSR are already present; skipping colab_setup.py.")

still_missing = [name for name in REQUIRED_SPECS if importlib.util.find_spec(name) is None]
if still_missing or not dossr_inference_py.is_file():
    raise RuntimeError(f"Setup incomplete. Missing specs: {still_missing}, DoSSR present: {dossr_inference_py.is_file()}")
print("CELL 05 READY")


---
## 7. Checkpoint Path Resolution & Optional SSD Cache (`CELL 06`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Verifies that `dossr_default.ckpt` exists on Google Drive. By default (`CACHE_CKPT_TO_LOCAL_SSD = False`), `ACTIVE_CKPT_PATH` points directly to the Drive file (since the OS page cache served the second load in `1.0 s` in earlier logs). If `CACHE_CKPT_TO_LOCAL_SSD = True`, copies the checkpoint to `/content/cache_sc203/weights/dossr_default.ckpt` and verifies at runtime that `dst.stat().st_size == src.stat().st_size`.
- **Optional check:** Includes the commented-out `scripts/verify_assets.py` command if you ever need to re-verify SHA-256 hashes on Drive.
- **Dependencies:** Requires `CELL 01`.
- **Expected output:** Prints `ACTIVE_CKPT_PATH`, byte size, and `CELL 06 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 06 [IN-CELL RUN] — Resolve checkpoint path and optional SSD cache with runtime size check (Status: NOT TESTED)
import shutil

DRIVE_CKPT_PATH = ASSETS_DIR / "weights" / "dossr_default.ckpt"
if not DRIVE_CKPT_PATH.is_file():
    raise FileNotFoundError(f"Missing DoSSR checkpoint on Drive: {DRIVE_CKPT_PATH}")

src_size = DRIVE_CKPT_PATH.stat().st_size
if src_size <= 0:
    raise RuntimeError(f"Drive checkpoint has invalid size ({src_size} bytes): {DRIVE_CKPT_PATH}")

if CACHE_CKPT_TO_LOCAL_SSD:
    local_ckpt = Path("/content/cache_sc203/weights/dossr_default.ckpt")
    local_ckpt.parent.mkdir(parents=True, exist_ok=True)
    if not local_ckpt.is_file() or local_ckpt.stat().st_size != src_size:
        print(f"Copying checkpoint to local SSD: {local_ckpt} ...")
        shutil.copyfile(DRIVE_CKPT_PATH, local_ckpt)
    dst_size = local_ckpt.stat().st_size
    if dst_size != src_size:
        local_ckpt.unlink(missing_ok=True)
        raise RuntimeError(
            f"Local checkpoint size mismatch after copy: dst={dst_size} vs src={src_size}"
        )
    ACTIVE_CKPT_PATH = local_ckpt
    print(f"Using local SSD checkpoint (verified {dst_size} bytes): {ACTIVE_CKPT_PATH}")
else:
    ACTIVE_CKPT_PATH = DRIVE_CKPT_PATH
    print(f"Using Drive checkpoint ({src_size} bytes): {ACTIVE_CKPT_PATH}")

# Optional full SHA-256 asset verification (uncomment only if assets on Drive changed):
# !{sys.executable} -u /content/SC203/scripts/verify_assets.py --assets-dir /content/drive/MyDrive/lemmepass --log-dir /content/drive/MyDrive/lemmepass/logs

print("CELL 06 READY")


---
## 8. GPU SDPA Numerical Equivalence Check (`CELL 07`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Runs `scripts/check_attention_sdpa.py --device cuda` on the GPU to compare:
  1. DoSSR's original `einsum` `CrossAttention` vs. PyTorch SDPA (`2 x 4 x 16 x 8`).
  2. DoSSR's original `bmm` VAE `AttnBlock` vs. single-head PyTorch SDPA on a tiny shape (`2 x 16 x 6 x 5`) and a larger shape (`1 x 512 x 64 x 64`), both kernel-only and through the patched `AttnBlock.forward`.
- **Why it is necessary:** Confirms that both runtime SDPA patches match the upstream attention math within `rtol=0.01, atol=0.01` in FP16 on the active CUDA GPU before running full diffusion inference.
- **Dependencies:** Requires `CELL 02`, `CELL 03`, and `CELL 05`. Uses GPU (`cuda`).
- **Expected output:** Logs 5 `passed` lines (`CrossAttention`, `VAE AttnBlock` tiny kernel & forward, `VAE AttnBlock` `1x512x64x64` kernel & forward) with `device=cuda`, and prints `CELL 07 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 07 [IN-CELL RUN] — Run CrossAttention and VAE AttnBlock SDPA check on GPU (Status: NOT TESTED)
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "check_attention_sdpa.py"),
        "--device",
        "cuda",
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)
print("CELL 07 READY")


---
## 9. Stage 1-Image FP16 Memory Profile Config (`CELL 08`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Runs `clean_corrupt_pngs()` on the dedicated output folder `OUTPUT_BASE / "baseline-smoke-vae-sdpa-fp16"`.
  2. Calls `scripts/prepare_baseline_smoke.py` to stage the first RealSR image (`Canon_001_LR4.png`) into `/content/realsr_smoke_1img` and write `/content/baseline_smoke_vae_sdpa_fp16.yaml`.
  3. Updates `checkpoint` in `/content/baseline_smoke_vae_sdpa_fp16.yaml` to `ACTIVE_CKPT_PATH` and sets `precision: fp16`.
- **Why it is necessary:** Prepares an isolated 1-image config with a fresh `output_root` so the post-VAE-patch run does not collide with earlier manifests.
- **Dependencies:** Requires `CELL 01`–`CELL 06`.
- **Expected output:** Prints `SMOKE CONFIG READY` and `CELL 08 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 08 [IN-CELL RUN] — Clean corrupt PNGs and stage 1-image FP16 config (Status: NOT TESTED)
import subprocess
import sys
import yaml

SMOKE_1IMG_DIR = Path("/content/realsr_smoke_1img")
SMOKE_1IMG_YAML = Path("/content/baseline_smoke_vae_sdpa_fp16.yaml")
SMOKE_1IMG_OUT_ROOT = OUTPUT_BASE / "baseline-smoke-vae-sdpa-fp16"

clean_corrupt_pngs(SMOKE_1IMG_OUT_ROOT)

# Remove stale temporary YAML if present so prepare_baseline_smoke.py writes a fresh copy
SMOKE_1IMG_YAML.unlink(missing_ok=True)

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "prepare_baseline_smoke.py"),
        "--config",
        str(REPO_DIR / "configs" / "baseline_realsr.yaml"),
        "--image-count",
        "1",
        "--smoke-dir",
        str(SMOKE_1IMG_DIR),
        "--config-out",
        str(SMOKE_1IMG_YAML),
        "--output-root",
        str(SMOKE_1IMG_OUT_ROOT),
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)

cfg = yaml.safe_load(SMOKE_1IMG_YAML.read_text(encoding="utf-8"))
cfg["checkpoint"] = str(ACTIVE_CKPT_PATH)
cfg["precision"] = "fp16"
SMOKE_1IMG_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

print("Config written:", SMOKE_1IMG_YAML)
print("Output root   :", SMOKE_1IMG_OUT_ROOT)
print("CELL 08 READY")


---
## 10. Run 1-Image FP16 Memory Profile (`CELL 09`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:** Executes `bash scripts/run_baseline.sh --config /content/baseline_smoke_vae_sdpa_fp16.yaml --fp16 --profile-memory`, saves console output to `/content/baseline_smoke_vae_sdpa_fp16_console.log`, verifies the required log lines (`enabling CUDA float16 autocast`, `using PyTorch scaled_dot_product_attention for vanilla CrossAttention`, and `using PyTorch scaled_dot_product_attention for vanilla VAE AttnBlock`), and prints a comparison against the pre-VAE-patch FP16 (`36.04 GiB`) and FP32 (`49.29 GiB`) baselines.
- **Why it is necessary:** Directly tests Task 1 from `context.md`: measuring how much the VAE `AttnBlock` SDPA patch reduces the `36.04 GiB` FP16 peak.
- **Dependencies:** Requires `CELL 08`. Uses GPU (`cuda`).
- **Expected output:** Prints memory profile lines, comparison table, and `CELL 09 READY`.
- **Troubleshooting:** If it fails, send back the last 50 lines of `/content/baseline_smoke_vae_sdpa_fp16_console.log` and the full traceback.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 09 [IN-CELL RUN] — Run 1-image FP16 memory profile and compare against 36.04 GiB (Status: NOT TESTED)
import re
import subprocess

SMOKE_CONSOLE_LOG = Path("/content/baseline_smoke_vae_sdpa_fp16_console.log")
clean_corrupt_pngs(SMOKE_1IMG_OUT_ROOT)

cmd = (
    f"cd {REPO_DIR} && set -o pipefail && "
    f"bash scripts/run_baseline.sh --config {SMOKE_1IMG_YAML} --fp16 --profile-memory "
    f"2>&1 | tee {SMOKE_CONSOLE_LOG}"
)
ret = subprocess.run(["bash", "-c", cmd], check=False)
if ret.returncode != 0:
    raise RuntimeError(
        f"1-image FP16 memory profile failed with exit code {ret.returncode}. "
        f"Inspect {SMOKE_CONSOLE_LOG}"
    )

log_text = SMOKE_CONSOLE_LOG.read_text(encoding="utf-8")
for required_marker in (
    "enabling CUDA float16 autocast",
    "using PyTorch scaled_dot_product_attention for vanilla CrossAttention",
    "using PyTorch scaled_dot_product_attention for vanilla VAE AttnBlock",
):
    if required_marker not in log_text:
        raise RuntimeError(f"Missing expected compatibility marker in log: {required_marker!r}")

after_sampler = re.search(
    r"DoSSR memory profile \| after sampler \|.*?peak_allocated=([0-9.]+) GiB \| peak_reserved=([0-9.]+) GiB",
    log_text,
)
sampler_time = re.search(r"DoSSR sampler profile \| elapsed=([0-9.]+)s", log_text)

print("\n=== 1-Image Memory Profile Comparison ===")
print("Baseline FP32 (CrossAttn SDPA only) : peak_allocated=49.29 GiB | peak_reserved=60.48 GiB | sampler=18.17 s")
print("Baseline FP16 (CrossAttn SDPA only) : peak_allocated=36.04 GiB | peak_reserved=52.70 GiB | sampler=4.61 s")
if after_sampler:
    peak_alloc = float(after_sampler.group(1))
    peak_res = float(after_sampler.group(2))
    s_time = sampler_time.group(1) if sampler_time else "n/a"
    print(
        f"Observed FP16 (+ VAE AttnBlock SDPA): "
        f"peak_allocated={peak_alloc:.2f} GiB | peak_reserved={peak_res:.2f} GiB | sampler={s_time} s"
    )
else:
    print("Note: Could not parse 'after sampler' profile line (if image was skipped by cache, run with a fresh output_root).")
print("CELL 09 READY")


---
## 11. Output Inspection, Pre-Patch Reference Diff & Disk Status JSON (`CELL 10`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Runs `scripts/inspect_output.py` on `Canon_001_LR4_0.png` (`expected_size=2048`, checks no NaN/Inf, not all-black, not all-white) and saves `Canon_001_LR4_compare.png` (bicubic left, DoSSR right).
  2. Displays `Canon_001_LR4_compare.png` inline for human visual inspection.
  3. If a pre-VAE-patch `Canon_001_LR4_0.png` exists (in `/content/outputs/baseline-smoke-l4-fp16/seed_42/` or already backed up on Drive), copies it to `OUTPUT_BASE / "reference_pre_vae_patch" / "seed_42" / "Canon_001_LR4_0.png"` and prints `max_abs_diff` and `mean_abs_diff` in 8-bit pixel levels (`0..255`) using `float32` arrays. **This pre- vs. post-patch diff is informational only and is not a pass/fail gate.**
  4. Writes `cell10_inspection_status.json` to disk with `passed=True`, `git_commit`, `timestamp_utc`, and `checked_image_paths`.
- **Dependencies:** Requires `CELL 09`. No GPU usage.
- **Expected output:** `OUTPUT INSPECTION PASSED`, inline comparison image, optional pre- vs. post-patch pixel diff in 8-bit levels, and `CELL 10 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 10 [IN-CELL RUN] — Inspect output, compare with pre-patch reference, write status JSON (Status: NOT TESTED)
import json
import shutil
import subprocess
import sys
import numpy as np
from IPython.display import display
from PIL import Image

INPUT_IMG_1 = SMOKE_1IMG_DIR / "Canon_001_LR4.png"
OUTPUT_IMG_42 = SMOKE_1IMG_OUT_ROOT / "seed_42" / "Canon_001_LR4_0.png"
COMPARE_IMG_42 = SMOKE_1IMG_OUT_ROOT / "seed_42" / "Canon_001_LR4_compare.png"

subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts" / "inspect_output.py"),
        "--input",
        str(INPUT_IMG_1),
        "--output",
        str(OUTPUT_IMG_42),
        "--comparison-output",
        str(COMPARE_IMG_42),
        "--expected-size",
        "2048",
        "--seed",
        "42",
        "--overwrite",
        "--log-dir",
        str(LOG_DIR),
    ],
    check=True,
)

# Copy pre-patch output to Drive as reference if available, and compute informational diff
PRE_PATCH_REF_DRIVE = OUTPUT_BASE / "reference_pre_vae_patch" / "seed_42" / "Canon_001_LR4_0.png"
PRE_PATCH_CANDIDATES = [
    Path("/content/outputs/baseline-smoke-l4-fp16/seed_42/Canon_001_LR4_0.png"),
    ASSETS_DIR / "outputs_backup" / "baseline-smoke-l4-fp16" / "seed_42" / "Canon_001_LR4_0.png",
    OUTPUT_BASE / "baseline-smoke-l4-fp16" / "seed_42" / "Canon_001_LR4_0.png",
]
if not PRE_PATCH_REF_DRIVE.is_file():
    for candidate in PRE_PATCH_CANDIDATES:
        if candidate.is_file():
            PRE_PATCH_REF_DRIVE.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidate, PRE_PATCH_REF_DRIVE)
            print(f"Copied pre-VAE-patch output to Drive reference: {PRE_PATCH_REF_DRIVE}")
            break

pre_vs_post_diff = None
if PRE_PATCH_REF_DRIVE.is_file():
    with Image.open(PRE_PATCH_REF_DRIVE) as im_pre, Image.open(OUTPUT_IMG_42) as im_post:
        arr_pre = np.asarray(im_pre.convert("RGB"), dtype=np.float32)
        arr_post = np.asarray(im_post.convert("RGB"), dtype=np.float32)
    abs_diff = np.abs(arr_post - arr_pre)
    max_diff = float(abs_diff.max())
    mean_diff = float(abs_diff.mean())
    pre_vs_post_diff = {"max_abs_diff_px": max_diff, "mean_abs_diff_px": mean_diff}
    print(
        f"Informational diff (pre-VAE-patch vs post-VAE-patch, 8-bit pixel levels 0..255): "
        f"max_abs_diff={max_diff:.4f} | mean_abs_diff={mean_diff:.4f}"
    )
else:
    print("Note: Pre-VAE-patch Canon_001_LR4_0.png not found; skipping pre-vs-post pixel diff.")

cell10_payload = {
    "passed": True,
    "git_commit": GIT_COMMIT_FULL,
    "timestamp_utc": utc_now_iso(),
    "checked_image_paths": [str(INPUT_IMG_1), str(OUTPUT_IMG_42), str(COMPARE_IMG_42)],
    "pre_vs_post_vae_patch_diff": pre_vs_post_diff,
}
CELL10_STATUS_JSON.write_text(json.dumps(cell10_payload, indent=2) + "\n", encoding="utf-8")
(OUTPUT_BASE / "cell10_inspection_status.json").write_text(
    json.dumps(cell10_payload, indent=2) + "\n", encoding="utf-8"
)

print("Displaying side-by-side comparison (Bicubic left | DoSSR right):")
with Image.open(COMPARE_IMG_42) as cmp_img:
    display(cmp_img.resize((1024, 512)))
print(f"Wrote status file: {CELL10_STATUS_JSON}")
print("CELL 10 READY")


---
## 12. Resume Path Test & 1-Image Seed Sensitivity Control (`CELL 11`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. **Resume Path Check:** Re-runs `scripts/run_baseline.sh --config /content/baseline_smoke_vae_sdpa_fp16.yaml --fp16` on the already-completed `SMOKE_1IMG_OUT_ROOT` and verifies `run_manifest.json` does not conflict and logs `OUTPUT CACHE | hits=1 | misses=0`.
  2. **Same-Seed Control (`seed 42-repeat`):** Writes a new YAML config (`/content/baseline_smoke_repeat42_fp16.yaml`) with `seed: 42` and a separate `output_root` (`OUTPUT_BASE / "seed-sanity-repeat42-fp16"`), cleans corrupt PNGs in that folder, and runs `bash scripts/run_baseline.sh --config /content/baseline_smoke_repeat42_fp16.yaml --fp16`.
  3. **Cross-Seed Run (`seed 43`):** Writes a new YAML config (`/content/baseline_smoke_seed43_fp16.yaml`) with `seed: 43` and a separate `output_root` (`OUTPUT_BASE / "seed-sanity-seed43-fp16"`), cleans corrupt PNGs in that folder, and runs `bash scripts/run_baseline.sh --config /content/baseline_smoke_seed43_fp16.yaml --fp16`.
  4. **Float32 Pixel Diff & Gate Rule:** Loads the three outputs (`42`, `42-repeat`, `43`) as `np.float32` arrays (never subtracting `uint8` directly) and prints `max_abs_diff` and `mean_abs_diff` in 8-bit pixel levels (`0..255`).
     - Gate rule: If `mean_abs_diff(42 vs 42-repeat) == 0`, passes when `mean_abs_diff(42 vs 43) > 0`. Otherwise passes when `mean_abs_diff(42 vs 43) >= SEED_SPREAD_MIN_RATIO * mean_abs_diff(42 vs 42-repeat)` (where `SEED_SPREAD_MIN_RATIO = 3.0`).
     - Writes `cell11_seed_sanity_status.json` (with `passed`, `git_commit`, `timestamp_utc`, `checked_image_paths`, and diff numbers) **before** raising an error if the gate fails.
- **Dependencies:** Requires `CELL 10`. Uses GPU (`cuda`).
- **Expected output:** Prints resume check result, `42 vs 42-repeat` diffs, `42 vs 43` diffs, status JSON path, and `CELL 11 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 11 [IN-CELL RUN] — Test resume path, run 42-repeat and 43, verify seed spread gate (Status: NOT TESTED)
import json
import subprocess
import numpy as np
import yaml
from PIL import Image

# 1. Test the resume path on SMOKE_1IMG_OUT_ROOT (seed 42 already finished in CELL 09)
print("=== Step 1: Testing resume path on existing seed_42 output ===")
clean_corrupt_pngs(SMOKE_1IMG_OUT_ROOT)
resume_proc = subprocess.run(
    ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(SMOKE_1IMG_YAML), "--fp16"],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
    check=False,
)
print(resume_proc.stdout)
if resume_proc.returncode != 0:
    raise RuntimeError(
        f"Resume path failed (possible run_manifest.json conflict):\n{resume_proc.stdout}\n{resume_proc.stderr}"
    )
if "OUTPUT CACHE | hits=1 | misses=0" not in resume_proc.stdout:
    raise RuntimeError("Resume check did not report 'OUTPUT CACHE | hits=1 | misses=0'.")
print("Resume check passed: manifest matched and existing PNG was reused (hits=1, misses=0).")

# 2. Same-seed control: run seed 42 a second time into a new output_root with --fp16
REPEAT42_OUT_ROOT = OUTPUT_BASE / "seed-sanity-repeat42-fp16"
REPEAT42_YAML = Path("/content/baseline_smoke_repeat42_fp16.yaml")
clean_corrupt_pngs(REPEAT42_OUT_ROOT)

base_cfg = yaml.safe_load(SMOKE_1IMG_YAML.read_text(encoding="utf-8"))
repeat42_cfg = {
    **base_cfg,
    "seed": 42,
    "precision": "fp16",
    "output_root": str(REPEAT42_OUT_ROOT),
}
REPEAT42_YAML.write_text(yaml.safe_dump(repeat42_cfg, sort_keys=False), encoding="utf-8")

print("\n=== Step 2: Running same-seed control (seed 42 repeat) into new output_root ===")
subprocess.run(
    ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(REPEAT42_YAML), "--fp16"],
    cwd=str(REPO_DIR),
    check=True,
)

# 3. Cross-seed check: run seed 43 into a new output_root via a new YAML with seed: 43 and --fp16
SEED43_OUT_ROOT = OUTPUT_BASE / "seed-sanity-seed43-fp16"
SEED43_YAML = Path("/content/baseline_smoke_seed43_fp16.yaml")
clean_corrupt_pngs(SEED43_OUT_ROOT)

seed43_cfg = {
    **base_cfg,
    "seed": 43,
    "precision": "fp16",
    "output_root": str(SEED43_OUT_ROOT),
}
SEED43_YAML.write_text(yaml.safe_dump(seed43_cfg, sort_keys=False), encoding="utf-8")

print("\n=== Step 3: Running seed 43 into new output_root ===")
subprocess.run(
    ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(SEED43_YAML), "--fp16"],
    cwd=str(REPO_DIR),
    check=True,
)

# 4. Compute diffs in float32 (8-bit pixel levels 0..255; never subtract uint8 directly)
OUTPUT_IMG_42_REPEAT = REPEAT42_OUT_ROOT / "seed_42" / "Canon_001_LR4_0.png"
OUTPUT_IMG_43 = SEED43_OUT_ROOT / "seed_43" / "Canon_001_LR4_0.png"

for p in (OUTPUT_IMG_42, OUTPUT_IMG_42_REPEAT, OUTPUT_IMG_43):
    if not p.is_file():
        raise FileNotFoundError(f"Expected output image missing: {p}")

with Image.open(OUTPUT_IMG_42) as im42, Image.open(OUTPUT_IMG_42_REPEAT) as im42r, Image.open(OUTPUT_IMG_43) as im43:
    a42 = np.asarray(im42.convert("RGB"), dtype=np.float32)
    a42r = np.asarray(im42r.convert("RGB"), dtype=np.float32)
    a43 = np.asarray(im43.convert("RGB"), dtype=np.float32)

diff_same = np.abs(a42 - a42r)
diff_cross = np.abs(a42 - a43)

max_same = float(diff_same.max())
mean_same = float(diff_same.mean())
max_cross = float(diff_cross.max())
mean_cross = float(diff_cross.mean())

print("\n=== Step 4: 1-Image Seed Sensitivity Results (8-bit pixel levels 0..255) ===")
print(f"Same-seed control (42 vs 42-repeat): max_abs_diff={max_same:.4f} | mean_abs_diff={mean_same:.4f}")
print(f"Cross-seed spread (42 vs 43)       : max_abs_diff={max_cross:.4f} | mean_abs_diff={mean_cross:.4f}")

if mean_same == 0.0:
    gate_passed = mean_cross > 0.0
    gate_rule_desc = "mean_same == 0.0 -> require mean_cross > 0.0"
else:
    gate_passed = mean_cross >= (SEED_SPREAD_MIN_RATIO * mean_same)
    gate_rule_desc = f"mean_cross >= {SEED_SPREAD_MIN_RATIO} * mean_same ({SEED_SPREAD_MIN_RATIO * mean_same:.4f})"

cell11_payload = {
    "passed": bool(gate_passed),
    "git_commit": GIT_COMMIT_FULL,
    "timestamp_utc": utc_now_iso(),
    "seed_spread_min_ratio": SEED_SPREAD_MIN_RATIO,
    "gate_rule": gate_rule_desc,
    "checked_image_paths": [
        str(OUTPUT_IMG_42),
        str(OUTPUT_IMG_42_REPEAT),
        str(OUTPUT_IMG_43),
    ],
    "diff_42_vs_42_repeat_px": {"max_abs_diff": max_same, "mean_abs_diff": mean_same},
    "diff_42_vs_43_px": {"max_abs_diff": max_cross, "mean_abs_diff": mean_cross},
}
CELL11_STATUS_JSON.write_text(json.dumps(cell11_payload, indent=2) + "\n", encoding="utf-8")
(OUTPUT_BASE / "cell11_seed_sanity_status.json").write_text(
    json.dumps(cell11_payload, indent=2) + "\n", encoding="utf-8"
)
print(f"Wrote status file: {CELL11_STATUS_JSON}")

if not gate_passed:
    raise RuntimeError(
        f"SEED SANITY GATE FAILED ({gate_rule_desc}): "
        f"42 vs 42-repeat (max={max_same:.4f}, mean={mean_same:.4f}) vs "
        f"42 vs 43 (max={max_cross:.4f}, mean={mean_cross:.4f}). "
        "Stopping before multi-seed study."
    )

print("SEED SANITY GATE PASSED")
print("CELL 11 READY")


---
## 13. Stage & Run 5-Image Baseline Batch (`CELL 12`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Checks `RUN_SEED_STUDY`. If `False` (the default), skips cleanly without running.
  2. When `RUN_SEED_STUDY = True`, reads `cell10_inspection_status.json` and `cell11_seed_sanity_status.json` **from disk**, verifies `"passed" is True` in both, and verifies `"git_commit"` in both matches the current `HEAD` commit hash (`git rev-parse HEAD`). Refuses to run on any mismatch.
  3. Cleans corrupt PNGs in `OUTPUT_BASE / "baseline-realsr-5img-vae-sdpa"`, stages the 5 lexically first RealSR images (`Canon_001` to `Canon_005`) into `/content/realsr_batch_5`, writes `/content/baseline_realsr_5img_vae_sdpa.yaml`, and runs `bash scripts/run_baseline.sh --config /content/baseline_realsr_5img_vae_sdpa.yaml --fp16`.
- **Dependencies:** Requires `CELL 10` and `CELL 11` status JSONs on disk and `RUN_SEED_STUDY = True`. Uses GPU (`cuda`).
- **Expected output:** Skips cleanly when `RUN_SEED_STUDY = False`; otherwise runs the 5-image batch and prints `CELL 12 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 12 [IN-CELL RUN] — Disk-gated 5-image FP16 batch run (Status: NOT TESTED)
import json
import subprocess
import sys
import yaml


def verify_disk_gates_and_commit() -> None:
    current_head = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    for label, status_path in (
        ("CELL 10 inspection", CELL10_STATUS_JSON),
        ("CELL 11 seed sanity", CELL11_STATUS_JSON),
    ):
        if not status_path.is_file():
            raise FileNotFoundError(f"Missing required status JSON on disk for {label}: {status_path}")
        data = json.loads(status_path.read_text(encoding="utf-8"))
        if data.get("passed") is not True:
            raise RuntimeError(f"{label} did not pass according to {status_path}: {data}")
        recorded_commit = data.get("git_commit")
        if recorded_commit != current_head:
            raise RuntimeError(
                f"Git commit mismatch in {status_path}: recorded {recorded_commit} != current HEAD {current_head}"
            )
        for checked_path in data.get("checked_image_paths", []):
            if not Path(checked_path).is_file():
                raise FileNotFoundError(f"Checked image recorded in {status_path} is missing: {checked_path}")
    print(f"Disk gate check passed (HEAD={current_head[:7]}).")


BATCH_5IMG_DIR = Path("/content/realsr_batch_5")
BATCH_5IMG_YAML = Path("/content/baseline_realsr_5img_vae_sdpa.yaml")
BATCH_5IMG_OUT_ROOT = OUTPUT_BASE / "baseline-realsr-5img-vae-sdpa"

if not RUN_SEED_STUDY:
    print("RUN_SEED_STUDY is False; skipping CELL 12 (5-image batch). Set RUN_SEED_STUDY = True in CELL 01 after reviewing CELL 09-11.")
else:
    verify_disk_gates_and_commit()
    clean_corrupt_pngs(BATCH_5IMG_OUT_ROOT)
    BATCH_5IMG_YAML.unlink(missing_ok=True)

    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts" / "prepare_baseline_smoke.py"),
            "--config",
            str(REPO_DIR / "configs" / "baseline_realsr_5img.yaml"),
            "--source-dir",
            str(REALSR_DIR),
            "--image-count",
            "5",
            "--smoke-dir",
            str(BATCH_5IMG_DIR),
            "--config-out",
            str(BATCH_5IMG_YAML),
            "--output-root",
            str(BATCH_5IMG_OUT_ROOT),
            "--log-dir",
            str(LOG_DIR),
        ],
        check=True,
    )

    cfg5 = yaml.safe_load(BATCH_5IMG_YAML.read_text(encoding="utf-8"))
    cfg5["checkpoint"] = str(ACTIVE_CKPT_PATH)
    cfg5["precision"] = "fp16"
    BATCH_5IMG_YAML.write_text(yaml.safe_dump(cfg5, sort_keys=False), encoding="utf-8")

    subprocess.run(
        ["bash", str(REPO_DIR / "scripts" / "run_baseline.sh"), "--config", str(BATCH_5IMG_YAML), "--fp16"],
        cwd=str(REPO_DIR),
        check=True,
    )
    print("CELL 12 READY")


---
## 14. Run 5-Seed Robustness Study (`seeds 42..46`) & Verify Outputs (`CELL 13`)
- **Cell Type:** `IN-CELL RUN` (Execute inside this Google Colab notebook cell)
- **What it does:**
  1. Checks `RUN_SEED_STUDY`. If `False`, skips cleanly.
  2. When `RUN_SEED_STUDY = True`, re-verifies `cell10_inspection_status.json` and `cell11_seed_sanity_status.json` on disk (including `passed is True` and `git_commit == HEAD`).
  3. Cleans corrupt PNGs in `OUTPUT_BASE / "seed-study-realsr-5img-vae-sdpa"`, writes `/content/seed_study_realsr_5img_vae_sdpa.yaml` (pointing `baseline_config` to `/content/baseline_realsr_5img_vae_sdpa.yaml`, which has `precision: fp16`), and executes `scripts/run_seed_study.py`.
  4. Verifies that all 5 seed folders (`seed_42`..`seed_46`) contain 5 valid PNGs and a `run_manifest.json` with `"precision": "fp16"`.
- **Dependencies:** Requires `CELL 12` and `RUN_SEED_STUDY = True`. Uses GPU (`cuda`).
- **Expected output:** Skips cleanly when `RUN_SEED_STUDY = False`; otherwise prints per-seed counts (25 total PNGs across 5 seeds) and `CELL 13 READY`.
- **Verification Status:** `NOT TESTED`


In [ ]:
# CELL 13 [IN-CELL RUN] — Disk-gated 5-seed robustness study (seeds 42..46) (Status: NOT TESTED)
import json
import subprocess
import sys
import yaml

SEED_STUDY_OUT_ROOT = OUTPUT_BASE / "seed-study-realsr-5img-vae-sdpa"
SEED_STUDY_YAML = Path("/content/seed_study_realsr_5img_vae_sdpa.yaml")
STUDY_SEEDS = [42, 43, 44, 45, 46]

if not RUN_SEED_STUDY:
    print("RUN_SEED_STUDY is False; skipping CELL 13 (5-seed study). Set RUN_SEED_STUDY = True in CELL 01 when ready.")
else:
    verify_disk_gates_and_commit()
    if not BATCH_5IMG_YAML.is_file():
        raise FileNotFoundError(f"Run CELL 12 first to create {BATCH_5IMG_YAML}")

    clean_corrupt_pngs(SEED_STUDY_OUT_ROOT)

    study_cfg = {
        "baseline_config": str(BATCH_5IMG_YAML),
        "seeds": STUDY_SEEDS,
        "output_root": str(SEED_STUDY_OUT_ROOT),
        "log_dir": str(LOG_DIR),
    }
    SEED_STUDY_YAML.write_text(yaml.safe_dump(study_cfg, sort_keys=False), encoding="utf-8")

    subprocess.run(
        [
            sys.executable,
            "-u",
            str(REPO_DIR / "scripts" / "run_seed_study.py"),
            "--config",
            str(SEED_STUDY_YAML),
            "--baseline-config",
            str(BATCH_5IMG_YAML),
        ],
        cwd=str(REPO_DIR),
        check=True,
    )

    clean_corrupt_pngs(SEED_STUDY_OUT_ROOT)
    print("\n=== 5-Seed Study Output Verification ===")
    for s in STUDY_SEEDS:
        s_dir = SEED_STUDY_OUT_ROOT / f"seed_{s}"
        pngs = sorted(s_dir.glob("*.png"))
        manifest_file = s_dir / "run_manifest.json"
        if len(pngs) != 5 or not manifest_file.is_file():
            raise RuntimeError(f"Incomplete seed folder {s_dir}: pngs={len(pngs)}, manifest={manifest_file.is_file()}")
        manifest_data = json.loads(manifest_file.read_text(encoding="utf-8"))
        if manifest_data.get("precision") != "fp16":
            raise RuntimeError(f"Seed {s} manifest precision is not fp16: {manifest_data}")
        print(f"  seed_{s}: {len(pngs)} PNGs verified | precision={manifest_data.get('precision')}")
    print("CELL 13 READY")


---
## 15. Troubleshooting & What to Send Back
- **Cell Type:** `MARKDOWN`
- If any cell fails, **do not** set `RUN_SEED_STUDY = True` or start larger runs.
- Send back:
  1. The console output of the failing cell and the **last 50 lines** of the latest log in `/content/drive/MyDrive/lemmepass/logs/` (plus `/content/baseline_smoke_vae_sdpa_fp16_console.log` if `CELL 09` failed).
  2. The **full Python traceback**.
  3. The outputs of `CELL 03` (`nvidia-smi` and VRAM report) and `CELL 04` (dataset folder structure and input image size).
